<a href="https://colab.research.google.com/github/K-Pratham-Prabhu09/Prototype-Lab/blob/pytorch_tutorials/pyTorchLearning_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print(torch.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
  print("GPU Not Available, would run on CPU")

2.10.0+cu128
Tesla T4


In [3]:
# Create a Tensor:
a =  torch.empty(2,3)
type(a)
b = torch.ones(3,2)
print(f"Data: {b}")
b.shape

Data: tensor([[1., 1.],
        [1., 1.],
        [1., 1.]])


torch.Size([3, 2])

In [4]:
torch.manual_seed(20)
b = torch.rand(2,3)
print(b)
c = torch.tensor([[1,2,3],[4,5,6]])

tensor([[0.5615, 0.1774, 0.8147],
        [0.3295, 0.2319, 0.7832]])


### Exercise 1: The Vision Data Pipeline (Reshaping & The NumPy Bridge)
When downloading image datasets using standard Python libraries, images are often loaded as NumPy arrays. Your task is to build a preprocessing pipeline to make this data compatible with PyTorch's convolutional neural networks.

**The Task:**
1. Use NumPy to generate a mock dataset of 64 RGB images. Create a NumPy array of shape `(64, 128, 128, 3)` containing random integers between `0` and `255`.
2. Convert this NumPy array into a PyTorch tensor.
3. PyTorch strictly expects image batches in the format `(Batch_Size, Channels, Height, Width)`. Rearrange the dimensions of your tensor to match this requirement.
4. Neural networks prefer small float values. Convert the tensor's data type to `float32` and scale the pixel values to sit precisely between `0.0` and `1.0`.
5. Write a conditional check to move this final processed tensor to a GPU if one is available; otherwise, keep it on the CPU.


In [59]:
#1
import numpy as np

data = np.random.randint(0,255,size=(64, 128, 128, 3))

data_tensor = torch.from_numpy(data).to(dtype=torch.float32)
data_tensor = (data_tensor / 255.0).permute(0, 3, 1, 2)

if torch.cuda.is_available():
  print("GPU is available")
  device = 'cuda'
else:
  print("GPU is not available")
  device = 'cpu'

  data_tensor = data_tensor.to(device=device)

GPU is available


### Exercise 2: Hardcoding a Neural Network Forward Pass
Before relying on PyTorch's automated layers, it is essential to understand the raw linear algebra underneath. You will implement the forward pass of a 2-layer Multi-Layer Perceptron (MLP).

**The Task:**
1. Set a manual seed of `42` to ensure your random initializations are reproducible.
2. Create an input tensor $X$ of shape `(32, 100)` representing a batch of 32 samples with 100 features each. Use random values between 0 and 1.
3. Create the first layer's weights $W_1$ with shape `(100, 50)` and a bias vector $b_1$ of shape `(50)`.
4. Create the second layer's weights $W_2$ with shape `(50, 10)` and a bias vector $b_2$ of shape `(10)`.
5. Compute the hidden layer activations $H$. The formula is $H = XW_1 + b_1$. Apply a ReLU activation function to $H$ **in-place** to save memory.
6. Compute the final output predictions $Y = HW_2 + b_2$. Apply a Softmax activation across the columns to turn the outputs into probabilities.

**Hints:**
* Use `torch.matmul()` for the matrix multiplications. Notice how PyTorch automatically broadcasts the bias vectors across the batch size during addition.
* Remember that appending an underscore to a function (e.g., `.relu_()`) performs the operation in-place.
* For `torch.softmax()`, you will need to specify the `dim` parameter. Think about whether the probabilities should sum to 1 across the rows or the columns.

In [40]:
#1
torch.manual_seed(42)
X = torch.randn(32,100)
W1 = torch.randn(100,50)
b1 = torch.zeros(50)
b2 = torch.zeros(10)

W2 = torch.randn(50,10)
H = torch.matmul(X,W1)+b1
H.relu_()
Y = torch.matmul(H,W2)+b2
Y =Y.softmax(dim=1)
print(Y.sum(dim=1))


tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


### Exercise 3: Vectorized State-Value Update
In algorithmic environments, you often need to evaluate the expected returns of various states. Instead of using slow `for` loops to iterate through states, you can perform a Bellman Optimality update simultaneously across an entire state-space using tensor reduction operations.

**The Task:**
1. Generate a mock tensor $q_k(s, a)$ of shape `(500, 6)`. This represents the current action-values for 500 distinct states and 6 possible actions. Use standard normal distribution or random floats.
2. Calculate the greedy state-value function $v_k(s) = \max_a q_k(s, a)$. The resulting tensor must have a shape of `(500,)`, representing the maximum expected value for each state.
3. Extract the deterministic optimal policy $\pi_*(s) = \arg\max_a q_k(s, a)$. This should yield a `(500,)` tensor containing the integer index (0 through 5) of the best action for every state.
4. Assume taking the optimal action yields an immediate reward tensor $R$ of shape `(500,)`. Create $R$ full of `1.0`s. Calculate the update target: $Target = R + \gamma v_k(s)$, assuming a discount factor $\gamma = 0.99$.

**Hints:**
* When you call `torch.max()` and specify a dimension, PyTorch returns a namedtuple containing *both* the maximum values and the indices where those maximums were found. You can unpack this to solve steps 2 and 3 simultaneously.
* Step 4 is a straight-forward application of scalar and element-wise tensor math.

In [58]:
#3
q_k = torch.randn(500,6,dtype=torch.float32)
v_k,pi_star = torch.max(q_k,dim=1)
R = torch.full((500,),1)
gamma = 0.99
Target = R + gamma*v_k